# XGB: order parameter distribution at 20 % cholesterol (DOPC/CHOL = 80/20), split by XGB label

Histogram of the chain order parameter averaged over carbons C2-C17, both leaflets pooled:
all chains, chains classified L_o, and chains classified L_d. Same style as `S_avg_distribution_pure_vs_60`.

- Labels: `heavy_data_NO_GIT/data_arrays_by_chain_xgb/dopc_80_*_ft_last_17pt_prediction_labels.npy`, made by `XGB_save_classifier_data` (17-pt window).
- The classifier used order parameters time-averaged over N_POINT frames (convolution), giving one label per chain per
  averaged frame. Here the carbon-averaged order parameter is time-averaged over the same N_POINT frames, so every value
  has exactly one label. "All chains" = the same values without the labels (L_o + L_d = all chains).
- Order parameters are cached in `heavy_data_NO_GIT/feature_cache/`.
  The first run calculates them from the positions; after that they load in seconds.

Run cells 1-3 in order. Cell 4 makes the combined figure (references on top, 80/20 below) from the saved CSVs;
it needs `S_avg_distribution_pure_vs_60` to have been run once.

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

# ---- 1. import packages ----
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import AutoMinorLocator
from code_libraries.classified_analysis import order_parameters, load_leaflets, cached, running_mean

In [ ]:
# ---- 2. file paths and settings ----
MODEL   = "XGB"
N_POINT = 17                      # window used in XGB_save_classifier_data (must match the label file name)
SYSTEM  = "dopc_80"                # chi_C = 0.20
SYSTEM_LABEL = "DOPC/CHOL = 80/20"
N_FRAMES = 4000

label_dir = "../heavy_data_NO_GIT/data_arrays_by_chain_xgb"
cache_dir = "../heavy_data_NO_GIT/feature_cache"
out_dir   = "../heavy_data_NO_GIT/results/s_avg_distributions"
os.makedirs(out_dir, exist_ok=True)

chains   = ["sn1", "sn2"]
leaflets = ["lower", "upper"]       # label files: index 0 = lower leaflet, 1 = upper leaflet

bins = np.arange(-0.6, 0.8 + 0.01, 0.01)    # histogram bins, width 0.01
x_limits = (-0.2, 0.4)                       # x axis range of the plot

In [ ]:
# ---- 3. averaged order parameter per chain (convolved like the classifier input), split by label, histograms ----
counts = {}                        # (chain, "all" / "Lo" / "Ld") -> number of chains in each bin
positions = None                   # DOPC positions, loaded only if the order parameters are not cached yet

for chain in chains:
    labels = np.load(os.path.join(label_dir, f"{SYSTEM}_{chain}_ft_last_{N_POINT}pt_prediction_labels.npy"))
    labels = labels.astype(bool)                    # (2 leaflets, frames - N_POINT + 1, lipids), True = Lo
    s_all, s_lo, s_ld = [], [], []

    for k, leaflet in enumerate(leaflets):
        cache_file = os.path.join(cache_dir, f"{SYSTEM}_{chain}_{leaflet}_ops_0_{N_FRAMES}.npy")
        if not os.path.exists(cache_file) and positions is None:
            positions = load_leaflets(SYSTEM, 0, N_FRAMES)[:2]          # (lower, upper) DOPC positions
        ops = cached(cache_file, lambda: order_parameters(positions[k], chain))   # (frames, lipids, 16 carbons)

        s_avg = ops.mean(axis=2)                                  # average over carbons -> (frames, lipids)
        s_avg = running_mean(s_avg, N_POINT)                      # average over N_POINT frames, same windows as the labels
        lab = labels[k]                                           # -> (frames - N_POINT + 1, lipids)
        assert s_avg.shape == lab.shape, (s_avg.shape, lab.shape)

        s_all.append(s_avg.ravel())
        s_lo.append(s_avg[lab])
        s_ld.append(s_avg[~lab])

    for kind, s in (("all", s_all), ("Lo", s_lo), ("Ld", s_ld)):
        s = np.concatenate(s)                                     # both leaflets together
        counts[(chain, kind)] = np.histogram(s, bins=bins)[0]
        print(f"{chain} {kind:3s}: {s.size:8d} values, mean = {s.mean():.3f}")
    n_lo, n_all = counts[(chain, "Lo")].sum(), counts[(chain, "all")].sum()
    print(f"{chain}: fraction classified Lo = {n_lo / n_all:.3f}\n")

bin_centers = (bins[:-1] + bins[1:]) / 2

# save the histograms
table = pd.DataFrame({"S_avg": bin_centers})
for (chain, kind), c in counts.items():
    table[f"{chain}_{kind}"] = c
table.to_csv(os.path.join(out_dir, f"{MODEL}_{N_POINT}pt_{SYSTEM}_s_avg_hist.csv"), index=False)

In [ ]:
# ---- 4. combined figure: top = Pure DOPC vs DOPC/CHOL = 60/40, bottom = this system classified ----
# sn-1 and sn-2 chains are counted together. Reads the saved histograms only:
# s_avg_hist_raw.csv (from S_avg_distribution_pure_vs_60) and the CSV saved in cell 3.
ref = pd.read_csv(os.path.join(out_dir, "s_avg_hist_raw.csv"))
mix = pd.read_csv(os.path.join(out_dir, f"{MODEL}_{N_POINT}pt_{SYSTEM}_s_avg_hist.csv"))

# sn-1 + sn-2, in units of 10^5 chains
SCALE  = 1e5
pure   = (ref["pure_dopc_sn1"] + ref["pure_dopc_sn2"]) / SCALE
chol40 = (ref["dopc_60_sn1"]   + ref["dopc_60_sn2"])   / SCALE
all_ch = (mix["sn1_all"] + mix["sn2_all"]) / SCALE
lo     = (mix["sn1_Lo"]  + mix["sn2_Lo"])  / SCALE
ld     = (mix["sn1_Ld"]  + mix["sn2_Ld"])  / SCALE

plt.rcParams.update({"font.weight": "bold", "axes.labelweight": "bold", "axes.linewidth": 2.5,
                     "mathtext.default": "bf"})
LW = 3          # line width
Y_HEADROOM = 1.15   # space above the highest curve (1.0 = touches the top)

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(12, 16))

# top: the two reference systems
ax_top.plot(ref["S_avg"], pure,   "-", lw=LW, color="green", label="Pure DOPC")
ax_top.plot(ref["S_avg"], chol40, "-", lw=LW, color="red",   label="DOPC/CHOL = 60/40")
ax_top.fill_between(ref["S_avg"], np.minimum(pure, chol40), color="0.6", alpha=0.6, lw=0)

# bottom: this system, split by the model's labels
ax_bot.plot(mix["S_avg"], all_ch, "-", lw=LW, color="black", label="All chains")
ax_bot.plot(mix["S_avg"], lo,     "-", lw=LW, color="red",   label=r"$\mathbf{L_o}$ chains")
ax_bot.plot(mix["S_avg"], ld,     "-", lw=LW, color="green", label=r"$\mathbf{L_d}$ chains")
ax_bot.fill_between(mix["S_avg"], np.minimum(lo, ld), color="0.6", alpha=0.6, lw=0)
ax_bot.text(0.04, 0.95, f"{MODEL}\n{SYSTEM_LABEL}", transform=ax_bot.transAxes, fontsize=32,
            fontweight="bold", ha="left", va="top")

for ax in (ax_top, ax_bot):
    ax.set_xlim(*x_limits)
    ax.set_xlabel(r"$\mathbf{\langle S_{CH} \rangle}$", fontsize=28)
    ax.set_ylabel(r"Number of Chains ($\mathbf{10^5}$)", fontsize=28)
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="major", labelsize=26, direction="in", top=True, right=True, length=10, width=2.5)
    ax.tick_params(axis="both", which="minor", direction="in", top=True, right=True, length=5, width=1.8)
    y_top = ax.get_ylim()[1] * Y_HEADROOM
    ax.set_ylim(-0.03 * y_top, y_top)            # small gap below 0 keeps the corner tick labels apart

ax_top.legend(fontsize=32, frameon=False, loc="upper left")
ax_bot.legend(fontsize=32, frameon=False, loc="center left", bbox_to_anchor=(0, 0.62))   # (x, y) of the legend's left middle

plt.tight_layout(h_pad=1)          # h_pad: space between the two panels
plt.savefig(os.path.join(out_dir, f"{MODEL}_{N_POINT}pt_{SYSTEM}_vs_references_s_avg_hist.png"), dpi=300)
plt.show()